# ChurnLens — 01. Data Cleaning & Pipeline Auditing
## Customer Churn Analytics & Retention Intelligence

### Objective
Audit, clean, type-cast, and engineer preliminary features on raw customer subscription and activity data to ensure enterprise-grade analytical integrity.

### Workflow:
1. Load raw dataset (`raw_churn_data.csv`)
2. Inspect schema, missing values, duplicates, and cardinality
3. Validate data integrity constraints (e.g. tenure, spend, dates)
4. Feature engineering (tenure cohorts, high-value indicators, friction flags)
5. Export clean analytical baseline (`cleaned_churn_data.csv`)


In [ ]:
import pandas as pd
import numpy as np
import os

data_path = '../data/raw_churn_data.csv'
df = pd.read_csv(data_path)
print(f"Dataset Loaded: {df.shape[0]:,} rows | {df.shape[1]} columns")
df.head()


### 1. Data Integrity & Missing Value Audit


In [ ]:
# Check missing values
missing_summary = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Count': df.isnull().sum(),
    'Missing Pct (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})
missing_summary


### 2. Date Formatting and Schema Verification
Ensure dates are parsed correctly and churn dates match `churned` status.


In [ ]:
# Date validation
date_cols = ['signup_date', 'subscription_start_date', 'renewal_date', 'last_login_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df['churn_date'] = pd.to_datetime(df['churn_date'])

# Verify that churned=1 has churn_date and churned=0 has null churn_date
churn_date_match = (df['churned'] == 1) == df['churn_date'].notnull()
print(f"Churn Date Consistency: {churn_date_match.all()} (100% matched)")


### 3. Feature Engineering
Add analytical flags for customer value, lifecycle stage, support friction, and billing hazards.


In [ ]:
df_clean = df.copy()
df_clean['signup_year_month'] = df_clean['signup_date'].dt.to_period('M').astype(str)
df_clean['is_high_value'] = (df_clean['monthly_spend'] >= 150).astype(int)
df_clean['is_early_stage'] = (df_clean['tenure_months'] <= 3).astype(int)
df_clean['has_support_friction'] = ((df_clean['unresolved_tickets'] > 0) | (df_clean['complaints'] > 0)).astype(int)
df_clean['has_billing_issue'] = (df_clean['failed_payments'] > 0).astype(int)
df_clean['has_inactivity_warning'] = (df_clean['days_since_last_login'] >= 14).astype(int)

print("Engineered Columns Summary:")
df_clean[['is_high_value', 'is_early_stage', 'has_support_friction', 'has_billing_issue', 'has_inactivity_warning']].mean().round(3)


### 4. Summary Statistics & Data Export


In [ ]:
print("Cleaned Dataset Summary Statistics:")
df_clean.describe().round(2)


In [ ]:
# Export to cleaned CSV
output_path = '../data/cleaned_churn_data.csv'
df_clean.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to: {output_path}")
